# PTD-BR Pipeline v2 — Corpus de Planos de Transformacao Digital

Pipeline de extracao automatica dos PTDs do Governo Federal Brasileiro.
Processa ~130 PDFs de ~90 orgaos → corpus estruturado com proveniencia.

**Arquitetura VSM (Beer/Enderle):** Este pipeline e S1 (Operacoes).
- Estagio 1: Coleta (scraping gov.br + SHA-256 dedup)
- Estagio 2: Extracao (Docling + OCR fallback)
- Estagio 3: Col-map via LLM (Haiku identifica colunas)
- Estagio 4: Classificacao (Aho-Corasick + regex eixos)

**Tempo estimado:** ~15 min (setup) + ~5 min por orgao

## 1. Setup (deps de sistema + Python)

In [ ]:
# Deps de sistema (Tesseract OCR + poppler para pdf2image)
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por poppler-utils

# Python deps
!pip install -q docling docling-core pyahocorasick pytesseract pdf2image
!pip install -q requests beautifulsoup4 anthropic pandas

## 2. Clone do repositorio

In [ ]:
!git clone -b claude/analyze-test-repo-lessons-3mXrF \
  https://github.com/freirelucas/ptd-br-corpus-de-planos-de-transforma-o-digital.git ptd-br
%cd ptd-br
!ls -la *.py config/

## 3. Configurar API key (opcional)

Para usar LLM no col_map (Estagio 3), configure `ANTHROPIC_API_KEY`:
- No Colab: Secrets (icone de chave) → adicionar `ANTHROPIC_API_KEY`
- Sem chave: usa heuristica fallback (limitada — ver Analise Falacia 3)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('API key configurada via Colab Secrets')
except Exception:
    print('Sem API key — col_map usara heuristica fallback (limitada)')

## 4. Executar pipeline (1 orgao piloto)

Comece com 1 orgao para validar. Depois escale.

In [ ]:
from ptd_pipeline import run

# Piloto: AGU (1 orgao, max 3 PDFs)
# Altere siglas_filtro para testar outros orgaos
# Remova filtros para rodar todos: run()
summary = run(siglas_filtro=['AGU'], max_pdfs=3)

## 5. Visualizar corpus

In [ ]:
import json
import pandas as pd

corpus = json.loads(open('ptd_corpus/03_database/ptd_corpus_v2.json').read())
df = pd.json_normalize(corpus)

print(f'Total linhas: {len(df)}')
print(f'Colunas: {list(df.columns)}')
print()
display(df[['sigla','servico','produto','eixo','parse_flag']].head(20))

## 6. Sensor S3* — diagnostico do run

In [ ]:
summary = json.loads(open('ptd_corpus/03_database/ptd_run_summary.json').read())

print(f"pct_ok:          {summary['pct_ok']}%")
print(f"sem_produto:     {summary['sem_produto_pct']}%")
print(f"Orgaos:          {summary['orgaos_processados']}")
print(f"Col-map fontes:  {summary['col_map_fontes']}")
print(f"Col-map confianca media: {summary['col_map_confianca_media']}")
print()
if summary['top_unmatched_phrases']:
    print('Top 10 frases sem match no vocabulario:')
    for item in summary['top_unmatched_phrases'][:10]:
        print(f"  {item['count']:3d} x {item['frase']}")

## 7. Inspecionar proveniencia (rastreabilidade)

In [ ]:
# Proveniencia: cada linha rastreavel ao PDF original
prov_cols = [c for c in df.columns if 'proveniencia' in c]
print(f'Campos de proveniencia: {prov_cols}')
print()

sample = df.sample(min(10, len(df)))
for _, row in sample.iterrows():
    print(f"Sigla: {row['sigla']}")
    print(f"  Servico: {str(row.get('servico',''))[:60]}")
    print(f"  Produto: {row.get('produto','')}")
    print(f"  Flag: {row.get('parse_flag','')}")
    print(f"  PDF: {row.get('proveniencia.pdf_filename','')} pag {row.get('proveniencia.pagina','')}")
    print(f"  Raw: {str(row.get('proveniencia.raw_text',''))[:80]}")
    print(f"  Col-map: {row.get('proveniencia.col_map_fonte','')} (conf {row.get('proveniencia.col_map_confianca','')})")
    print()

## 8. Exportar CSV

In [ ]:
# Export CSV para pesquisadores
csv_path = 'ptd_corpus/03_database/ptd_corpus_v2.csv'
df.to_csv(csv_path, index=False)
print(f'CSV salvo: {csv_path} ({len(df)} linhas)')

# Download no Colab
try:
    from google.colab import files
    files.download(csv_path)
except: pass

## 9. Escalar (opcional)

Apos validar com 1 orgao, rode para todos:

In [ ]:
# CUIDADO: roda para TODOS os orgaos (~90). Demora ~1-2h.
# Descomente para executar:

# summary_full = run()